# **Regression: Stacking Regressor (The Final Ensemble)**

## **Justification of Algorithm Selection (The "Board of Directors")**

Stacking involves training a meta-model to intelligently combine the continuous predictions of several distinct base experts. The strength of this ensemble relies entirely on the **diversity** of its members. We must select models that make different types of mathematical errors, allowing the meta-model to learn when to trust which expert based on the patient's feature profile. 

We have curated the following "Elite Team" using the exact hyperparameter configurations that yielded the lowest RMSE in our previous independent optimization runs:
* **Linear Regression:** Our baseline mathematical champion. Highly stable, interpretable, and geometrically linear.
* **Random Forest Regressor:** Our variance-reduction expert. It uses a "wisdom of the crowd" approach with deep, diverse, macro-pruned trees.
* **XGBoost & Gradient Boosting (GBR):** Our advanced sequential boosting experts. They utilize highly optimized, shallow trees to iteratively correct residual errors.
* **AdaBoost Regressor:** Our adaptive boosting champion, utilizing weak learners to focus on the hardest-to-predict clinical cases.

### **The Excluded Models (Strategic Omissions)**
Strategic exclusion is just as important as inclusion. We intentionally rejected the following models to protect the computational integrity and real-time viability of the final pipeline:
* **Decision Tree Regressor:** Redundant. The Random Forest already represents the optimal, stabilized evolution of our Decision Tree structure.
* **K-Nearest Neighbors (K-NN):** Rejected due to the "lazy learner" bottleneck. It requires calculating distances against 80,000 historical records in real-time, creating unacceptable latency for a production-grade Streamlit application.
* **Support Vector Regressor (SVR):** Rejected due to extreme computational complexity ($O(n^3)$). Including it in a Stacking Regressor (which utilizes internal 3-fold cross-validation to generate training metadata) would result in unfeasible execution times spanning several days.

## **Preprocessing & Experiment Design**

Because our ensemble features a distance-dependent base model (Linear Regression) and the meta-model itself is typically a regularized linear model (Ridge Regression), feature scaling is strictly mandatory for the entire pipeline. While the tree-based algorithms are scale-invariant and will not be harmed by scaled inputs, the linear models require them to assign proper mathematical weights.

We define a tournament of **2 focused experiments**:
* **Standardized Champion Stack**: Evaluated on data scaled via `StandardScaler`.
* **Normalized Champion Stack**: Evaluated on data scaled via `MinMaxScaler`.

Following our established methodology, we **do not run Optuna or GridSearchCV on the meta-model**. Injecting heavily optimized base champions and then hyper-optimizing the meta-layer frequently triggers "Level-2 Overfitting" (where the meta-model simply memorizes the base models' training mistakes). We strictly log **both Train and Test metrics** to validate the generalization of our final architecture.

In [1]:
import pandas as pd
import numpy as np
import time
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Regression_Stacking")

# 2. Data Loading and Preparation
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")

categorical_cols = [
    'gender', 'ethnicity', 'smoking_status', 'education_level',
    'employment_status', 'age_groups', 'weight_status', 'income_level'
]

# Apply One-Hot Encoding
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Separate features and target 
# Drop classification targets to prevent data leakage!
X = df_final.drop(["diagnosed_diabetes", "diabetes_stage", "diabetes_risk_score"], axis=1)
y = df_final['diabetes_risk_score']

# Split data (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns
SEED = 42

def log_regression_metrics(y_tr_true, y_tr_pred, y_te_true, y_te_pred, duration):
    # Logs Train and Test metrics explicitly to monitor the Overfitting Gap
    # Train Partition Metrics
    mlflow.log_metric("rmse_train", mean_squared_error(y_tr_true, y_tr_pred) ** 0.5)
    mlflow.log_metric("mae_train", mean_absolute_error(y_tr_true, y_tr_pred))
    mlflow.log_metric("r2_train", r2_score(y_tr_true, y_tr_pred))
    
    # Test Partition Metrics
    mlflow.log_metric("rmse_test", mean_squared_error(y_te_true, y_te_pred) ** 0.5)
    mlflow.log_metric("mae_test", mean_absolute_error(y_te_true, y_te_pred))
    mlflow.log_metric("r2_test", r2_score(y_te_true, y_te_pred))
    
    mlflow.log_metric("fit_time", duration)

# ---------------------------------------------------------
# DEFINING THE CHAMPION MODELS (Fixed winners from previous runs)
# ---------------------------------------------------------

champion_xgb = XGBRegressor(
    n_estimators=200,
    learning_rate=0.07300190646052784,
    max_depth=6,
    random_state=SEED,
    n_jobs=-1
)

champion_rf = RandomForestRegressor(
    n_estimators=150,
    max_depth=25,
    min_samples_leaf=10,
    max_features=1.0,
    criterion='squared_error',
    random_state=SEED,
    n_jobs=-1
)

champion_ada = AdaBoostRegressor(
    n_estimators=177,
    learning_rate=1.3282165134621702,
    loss='square',
    random_state=SEED
)

champion_gb = GradientBoostingRegressor(
    n_estimators=220,
    learning_rate=0.05490747097522296,
    max_depth=6,
    subsample=0.7038013432002436,
    random_state=SEED
)

champion_lr = LinearRegression(
    fit_intercept=True
)

# List of base specialists for the Stack
champions_list = [
    ('xgb', champion_xgb),
    ('rf', champion_rf),
    ('ada', champion_ada),
    ('gb', champion_gb),
    ('lr', champion_lr)
]

# ---------------------------------------------------------
# 2 RUNS (Standardization vs Normalization)
# ---------------------------------------------------------
scalers = {
    "Standardization": StandardScaler(),
    "Normalization": MinMaxScaler()
}

for s_name, scaler_obj in scalers.items():
    with mlflow.start_run(run_name=f"Stacking_Reg_{s_name}_Champions"): 
        # Apply scaling to numerical features (Required for the Linear model inside the stack)
        X_train_scaled = X_train.copy()
        X_test_scaled = X_test.copy()
        X_train_scaled[num_cols] = scaler_obj.fit_transform(X_train[num_cols])
        X_test_scaled[num_cols] = scaler_obj.transform(X_test[num_cols])

        # Initialize Stacking with fixed champions and default RidgeCV meta-model
        stack_model = StackingRegressor(
            estimators=champions_list, 
            final_estimator=RidgeCV(), # Ridge with built-in Cross-Validation for the final weights
            cv=3, # Internal cross-validation to prevent leakage from base models
            n_jobs=-1
        )
        start_time = time.time()
        stack_model.fit(X_train_scaled, y_train)
        duration = time.time() - start_time
        
        # Explicit Predictions
        y_pred_train_stack = stack_model.predict(X_train_scaled)
        y_pred_test_stack = stack_model.predict(X_test_scaled)
        
        # MLflow logging
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("meta_model", "RidgeCV_Default")
        mlflow.log_param("optimization", "fixed_champions_team")
        mlflow.log_param("team_size", len(champions_list))
        mlflow.log_param("included_models", "XGB, RF, ADA, GB, LR")
            
        log_regression_metrics(y_train, y_pred_train_stack, y_test, y_pred_test_stack, duration)

2026/05/22 17:01:10 INFO mlflow.tracking.fluent: Experiment with name 'Regression_Stacking' does not exist. Creating a new experiment.


## Winner Run Selection (Priority Elimination Framework)

### Policy
A run is only eligible to win if it does NOT show evidence of overfitting or underfitting. Before applying the MAE/RMSE/R² decision rules, we require the Train→Test gaps to remain small enough to indicate acceptable generalization. Runs that memorize the training set or show a large Train/Test gap are disqualified regardless of metric rank.

### Selection Criteria (priority order)
1. **Generalization filter (mandatory):** runs with overfitting or underfitting are removed from consideration.
2. **Priority 1 (60%): Lowest MAE (Test)** — primary objective for regression accuracy.
3. **Priority 2 (30%): Lowest RMSE (Test)** — used to reject runs where RMSE grows disproportionately relative to MAE.
4. **Priority 3 (10%): Acceptable R² (Test)** — confirms explanatory quality.
5. **Tiebreaker: Lowest Fit Time** — if MAE, RMSE, and R² are effectively tied.

### Runs Summary

| Run | Scaler | MAE (Train) | MAE (Test) | RMSE (Train) | RMSE (Test) | R² (Train) | R² (Test) | Fit Time |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| Stacking_Reg_Standardization_Champions | Standardization | 0.16500 | 0.18594 | 0.21738 | 0.24513 | 0.99942 | 0.99927 | 252.55s |
| Stacking_Reg_Normalization_Champions | Normalization | 0.16486 | 0.18607 | 0.21738 | 0.24537 | 0.99942 | 0.99927 | 270.53s |

### Generalization Check (Test − Train)
- **Stacking_Reg_Standardization_Champions:** MAE gap = 0.18594 − 0.16500 = **+0.02094** and RMSE gap = 0.24513 − 0.21738 = **+0.02775** → PASS.
- **Stacking_Reg_Normalization_Champions:** MAE gap = 0.18607 − 0.16486 = **+0.02121** and RMSE gap = 0.24537 − 0.21738 = **+0.02799** → PASS.

### Overfitting / Underfitting Validation
- Neither run shows overfitting. The Train/Test gaps are small and stable for both MAE and RMSE.
- Neither run shows underfitting. Both Test R² values are extremely high, indicating strong explanatory power.
- There is no evidence of catastrophic RMSE growth relative to MAE.

### Step-by-Step Elimination
**Step 1 — Apply the generalization filter**
- Passing runs: both runs.

**Step 2 — Compare Test MAE (Priority 1 — 60%)**
- Stacking_Reg_Standardization_Champions: 0.18594
- Stacking_Reg_Normalization_Champions: 0.18607
- Lowest MAE: **Stacking_Reg_Standardization_Champions**.

**Step 3 — Compare Test RMSE (Priority 2 — 30%)**
- Stacking_Reg_Standardization_Champions: 0.24513
- Stacking_Reg_Normalization_Champions: 0.24537
- Stacking_Reg_Standardization_Champions remains the best choice.

**Step 4 — Check Test R² (Priority 3 — 10%)**
- Stacking_Reg_Standardization_Champions: 0.99927
- Stacking_Reg_Normalization_Champions: 0.99927
- The values are effectively tied, but Standardization stays slightly ahead on MAE and RMSE.

### Final Decision
**Winner: Stacking_Reg_Standardization_Champions**

**Justification:** `Stacking_Reg_Standardization_Champions` is the strongest run among those that pass the generalization filter. It has the lowest Test MAE, the lowest Test RMSE, and effectively tied Test R² with the normalization run. Fit time is not needed as a tiebreaker.

## Winner Hyperparameters
| Parameter | Value |
|---|---|
| **Scaler** | Standardization |
| **Meta-model** | RidgeCV_Default |
| **Team size** | 5 |
| **Included models** | XGB, RF, ADA, GB, LR |
| **Optimization** | fixed_champions_team |

## Overfitting / Underfitting Diagnosis
- Both runs show small Train→Test gaps on MAE and RMSE, so there is no disqualifying overfitting or underfitting under the current policy.
- Because the metrics are effectively tied on R², the slightly stronger Standardization run wins on the core regression metrics.